# Clase 055 — Feature engineering avanzado: target encoding + MICE

Dataset sintético de clasificación con 3 categóricas de alta cardinalidad (1000 niveles) + missing values.
Comparamos one-hot vs target encoding, y SimpleImputer vs MICE.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge

rng = np.random.default_rng(42)
np.random.seed(42)

## 1. Dataset sintético

1000 niveles por columna, target con señal en las categorías (cada nivel tiene una probabilidad latente).

In [ ]:
n_samples, n_levels = 5000, 1000

# Probabilidades latentes por nivel (lo que aprenderá target encoding)
latent_a = rng.beta(2, 2, n_levels)
latent_b = rng.beta(2, 2, n_levels)
latent_c = rng.beta(2, 2, n_levels)

cat_a = rng.integers(0, n_levels, n_samples)
cat_b = rng.integers(0, n_levels, n_samples)
cat_c = rng.integers(0, n_levels, n_samples)
num_1 = rng.normal(0, 1, n_samples)
num_2 = rng.normal(0, 1, n_samples)

logit = (latent_a[cat_a] - 0.5) + (latent_b[cat_b] - 0.5) + 0.3 * num_1 + 0.2 * num_2
p = 1 / (1 + np.exp(-3 * logit))
y = (rng.uniform(0, 1, n_samples) < p).astype(int)

df = pd.DataFrame({'cat_a': cat_a, 'cat_b': cat_b, 'cat_c': cat_c, 'num_1': num_1, 'num_2': num_2})
print('shape', df.shape, 'pos rate', y.mean().round(3))

## 2. Target encoding manual con smoothing bayesiano

$\text{enc}(c) = \frac{n_c \cdot \bar{y}_c + k \cdot \bar{y}_{\text{global}}}{n_c + k}$

In [ ]:
def target_encoding(x_train, y_train, x_test, smoothing=10):
    """Target encoding con smoothing bayesiano. Fitted en train, aplicado a ambos."""
    global_mean = y_train.mean()
    df_t = pd.DataFrame({'x': x_train, 'y': y_train})
    agg = df_t.groupby('x')['y'].agg(['mean', 'count'])
    enc = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    return (pd.Series(x_train).map(enc).fillna(global_mean).values,
            pd.Series(x_test).map(enc).fillna(global_mean).values)

def loo_target_encoding(x, y, smoothing=10):
    """Leave-one-out target encoding (sin leakage)."""
    global_mean = y.mean()
    df_t = pd.DataFrame({'x': x, 'y': y})
    agg = df_t.groupby('x')['y'].agg(['sum', 'count'])
    s = pd.Series(x).map(agg['sum']).values
    c = pd.Series(x).map(agg['count']).values
    return ((s - y) + smoothing * global_mean) / ((c - 1) + smoothing)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=42, stratify=y)

## 3. One-hot vs target encoding (LogReg + GBM)

In [ ]:
# Target encoding
X_train_te = X_train[['num_1', 'num_2']].copy()
X_test_te = X_test[['num_1', 'num_2']].copy()
for col in ['cat_a', 'cat_b', 'cat_c']:
    tr, te = target_encoding(X_train[col].values, y_train, X_test[col].values, smoothing=10)
    X_train_te[col + '_te'] = tr
    X_test_te[col + '_te'] = te

# One-hot (sparse para no reventar RAM con 3000 columnas)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_oh = ohe.fit_transform(X_train[['cat_a', 'cat_b', 'cat_c']])
X_test_oh = ohe.transform(X_test[['cat_a', 'cat_b', 'cat_c']])
print(f'one-hot shape: {X_train_oh.shape} ({X_train_oh.nnz} nnz)')
print(f'target enc shape: {X_train_te.shape}')

In [ ]:
results = {}

# LogReg + one-hot
lr = LogisticRegression(max_iter=300, random_state=42)
lr.fit(X_train_oh, y_train)
results['LogReg + OneHot'] = roc_auc_score(y_test, lr.predict_proba(X_test_oh)[:, 1])

# LogReg + target encoding
lr2 = LogisticRegression(max_iter=300, random_state=42)
lr2.fit(X_train_te, y_train)
results['LogReg + TargetEnc'] = roc_auc_score(y_test, lr2.predict_proba(X_test_te)[:, 1])

# GBM + target encoding
gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train_te, y_train)
results['GBM + TargetEnc'] = roc_auc_score(y_test, gbm.predict_proba(X_test_te)[:, 1])

print(pd.Series(results).round(4).to_string())

## 4. Missing values: MCAR vs MAR

- **MCAR** (missing completely at random): la probabilidad de NaN no depende de nada.
- **MAR** (missing at random): la probabilidad de NaN depende de otra feature observada.

In [ ]:
X_num = pd.DataFrame({
    'a': rng.normal(0, 1, 2000),
    'b': rng.normal(0, 1, 2000),
    'c': rng.normal(0, 1, 2000),
})
X_num['b'] = 0.7 * X_num['a'] + 0.3 * X_num['b']  # a y b correlacionadas
X_num['c'] = 0.5 * X_num['a'] + 0.5 * X_num['c']

# MCAR: 20% random
X_mcar = X_num.copy()
mask_mcar = rng.uniform(0, 1, X_num.shape) < 0.2
X_mcar[mask_mcar] = np.nan

# MAR: NaN en 'b' depende de 'a' (cuando a alto, b se pierde)
X_mar = X_num.copy()
X_mar.loc[X_mar['a'] > 0.5, 'b'] = np.nan

print('MCAR NaN ratio:', X_mcar.isna().mean().round(3).to_dict())
print('MAR  NaN ratio:', X_mar.isna().mean().round(3).to_dict())

## 5. SimpleImputer vs IterativeImputer (MICE)

Medimos el error de reconstrucción: imputado vs valor verdadero.

In [ ]:
def eval_imputer(X_true, X_with_nan, imputer):
    mask = X_with_nan.isna().values
    X_imp = imputer.fit_transform(X_with_nan)
    err = np.abs(X_imp[mask] - X_true.values[mask]).mean()
    return err

results = []
for name, X_nan in [('MCAR', X_mcar), ('MAR', X_mar)]:
    simple_mean = eval_imputer(X_num, X_nan, SimpleImputer(strategy='mean'))
    mice = eval_imputer(X_num, X_nan,
                        IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42))
    results.append({'tipo': name, 'SimpleImputer (mean)': simple_mean, 'MICE (BayesianRidge)': mice})

print(pd.DataFrame(results).round(4).to_string(index=False))
print('\nMICE gana fuerte en MAR porque usa la correlación a↔b para predecir b cuando falta.')

## 6. Sesgo de imputación simple

SimpleImputer comprime la varianza (todos los NaN → mismo valor); MICE preserva la distribución.

In [ ]:
X_imp_simple = pd.DataFrame(
    SimpleImputer(strategy='mean').fit_transform(X_mar), columns=X_num.columns)
X_imp_mice = pd.DataFrame(
    IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42).fit_transform(X_mar),
    columns=X_num.columns)

print('std verdadero:', X_num.std().round(3).to_dict())
print('std SimpleImp:', X_imp_simple.std().round(3).to_dict())
print('std MICE     :', X_imp_mice.std().round(3).to_dict())
print('\ncorr(a,b) verdadero:', X_num.corr().loc["a","b"].round(3))
print('corr(a,b) SimpleImp:', X_imp_simple.corr().loc["a","b"].round(3))
print('corr(a,b) MICE     :', X_imp_mice.corr().loc["a","b"].round(3))

## Ejercicios

1. Implementá CatBoost-style ordered target encoding y compará con smoothing.
2. Probá `KNNImputer` y agregálo a la tabla.
3. Aplicá MICE con `RandomForestRegressor` como estimador.

## Conclusiones

- Target encoding con smoothing gana fuerte cuando hay alta cardinalidad.
- LOO o CV evita el leakage clásico (cada sample no ve su propio target).
- MICE supera a SimpleImputer especialmente en MAR — explota correlaciones.
- SimpleImputer aplasta la varianza y subestima correlaciones.